In [15]:
import os
import time
import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm

# Шаг 1: список тикеров (без дублей)
raw_tickers = [
    'BTC', 'ETH', 'BNB', 'SOL', 'XRP', 'ADA', 'DOGE', 'AVAX', 'SHIB', 'DOT',
    'LINK', 'TRX', 'MATIC', 'UNI', 'LTC', 'ICP', 'NEAR', 'APT', 'FIL', 'ATOM',
    'XLM', 'STX', 'HBAR', 'IMX', 'GRT', 'INJ', 'OP', 'RNDR', 'THETA', 'EGLD',
    'TIA', 'SUI', 'SEI', 'FET', 'ALGO', 'FLOW', 'VET', 'AAVE', 'MKR', 'CRV',
    'RUNE', 'QNT', 'LDO', 'PENDLE', 'SAND', 'MANA', 'AXS', 'GALA', 'BEAM', 'ENJ',
    'APE', 'CHZ', 'GMT', 'COMP', 'SNX', 'YFI', 'JUP', 'PYTH', 'JTO', 'RBN',
    'ENS', 'XMR', 'ZEC', 'DASH', 'KAS', 'MINA', 'WOO', 'ORDI', 'SATS', 'BTT',
    'HOT', 'ANKR', 'RVN', 'WAVES', 'BAT', 'ZIL', 'QTUM', 'ONT', 'NEO', 'IOST',
    'OMG', 'USDT', 'USDC', 'DAI', 'FDUSD', 'TUSD', 'BUSD', 'USDD', 'FRAX', 'LUSD',
    'BCH', 'ETC', 'XEC', 'ASTR', 'AR', 'GNO', 'CAKE', 'FTM', 'LUNC', 'FLOKI',
    'BONK', 'WIF', 'RAY', 'DYDX'
]

unique_tickers = sorted(set(raw_tickers))
yahoo_tickers = [f"{t}-USD" for t in unique_tickers]
print(f"🎯 Сформирован пул из {len(yahoo_tickers)} уникальных криптовалют.")

# Шаг 2: скачивание с улучшенной обработкой ошибок и фиксацией Мультииндекса
dict_day, dict_hour, dict_minute = {}, {}, {}
RATE_LIMIT_DELAY = 0.6

print("\n🚀 Скачивание данных из Yahoo Finance...")
for y_ticker in tqdm(yahoo_tickers):
    pure_ticker = y_ticker.split('-')[0]
    time.sleep(RATE_LIMIT_DELAY)

    try:
        # Качаем данные. auto_adjust=True гарантирует наличие чистой колонки 'Close'
        df_day = yf.download(y_ticker, period="30d", interval="1d", progress=False, auto_adjust=True)
        df_hour = yf.download(y_ticker, period="3d", interval="1h", progress=False, auto_adjust=True)
        df_min = yf.download(y_ticker, period="1d", interval="1m", progress=False, auto_adjust=True)

        if not df_day.empty and not df_hour.empty and not df_min.empty:
            # ИСПРАВЛЕНИЕ: Извлекаем чистую одномерную серию, сбрасывая Мультииндекс yfinance
            dict_day[pure_ticker] = pd.Series(df_day['Close'].values.flatten(), index=df_day.index)
            dict_hour[pure_ticker] = pd.Series(df_hour['Close'].values.flatten(), index=df_hour.index)
            dict_minute[pure_ticker] = pd.Series(df_min['Close'].values.flatten(), index=df_min.index)
    except Exception as e:
        print(f"⚠️ Ошибка для {y_ticker}: {e}")
        continue

# =====================================================================
# ШАГ 3: МЕТОДОЛОГИЧЕСКИ ЧИСТАЯ ОЧИСТКА И ВЫРАВНИВАНИЕ (УЛУЧШЕНО)
# =====================================================================
df_day_wide = pd.DataFrame(dict_day).sort_index()
df_hour_wide = pd.DataFrame(dict_hour).sort_index()
df_minute_wide = pd.DataFrame(dict_minute).sort_index()

if 'BTC' not in df_day_wide.columns or 'BTC' not in df_hour_wide.columns or 'BTC' not in df_minute_wide.columns:
    raise ValueError("⚠️ Данные Биткоина (BTC) не скачались. Проверьте соединение и повторите.")

ref_idx_day = df_day_wide['BTC'].index
ref_idx_hour = df_hour_wide['BTC'].index
ref_idx_min = df_minute_wide['BTC'].index

# 1. Выравниваем сетки строго по BTC
# 2. ffill(limit=...) заполняет только случайные микро-дыры внутри торгов (не более 3 точек подряд)
# 3. .bfill() ПОЛНОСТЬЮ УДАЛЕН, чтобы не создавать фальшивую историю в прошлом!
# 4. dropna(axis=1, how='any') ЖЕСТКО отсекает монеты без полной истории на всем промежутке
df_day_clean = df_day_wide.reindex(ref_idx_day).ffill(limit=2).dropna(axis=1, how='any')
df_hour_clean = df_hour_wide.reindex(ref_idx_hour).ffill(limit=3).dropna(axis=1, how='any')
df_minute_clean = df_minute_wide.reindex(ref_idx_min).ffill(limit=2).dropna(axis=1, how='any')

# Обрезаем минутные данные до последних 60 точек строго ПОСЛЕ выравнивания сеток
if len(df_minute_clean) >= 60:
    df_minute_clean = df_minute_clean.tail(60)

# Находим монеты, которые успешно выжили абсолютно во всех трех временных срезах
valid_tickers = list(set(df_day_clean.columns) & set(df_hour_clean.columns) & set(df_minute_clean.columns))

# Железно гарантируем наличие BTC
if 'BTC' not in valid_tickers:
    valid_tickers.append('BTC')

# Формируем финальные, методологически честные матрицы
df_day_final = df_day_clean[valid_tickers]
df_hour_final = df_hour_clean[valid_tickers]
df_minute_final = df_minute_clean[valid_tickers]

print(f"\n✅ Академически чистая валидация завершена.")
print(f"📈 Настоящую полную историю во всех 3 режимах подтвердили: {len(valid_tickers)} монет.")



🎯 Сформирован пул из 104 уникальных криптовалют.

🚀 Скачивание данных из Yahoo Finance...


  5%|▍         | 5/104 [00:05<01:37,  1.01it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['APT-USD']: YFPricesMissingError('possibly delisted; no price data found  (period=30d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['APT-USD']: YFPricesMissingError('possibly delisted; no price data found  (period=3d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['APT-USD']: YFPricesMissingError('possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")')
 20%|██        | 21/104 [00:21<01:22,  1.00it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['COMP-USD']: YFPricesMissingError('possibly delisted; no price data found  (period=30d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['COMP-USD']: YFPricesMissingError('possibly delisted; no price data fou


✅ Академически чистая валидация завершена.
📈 Настоящую полную историю во всех 3 режимах подтвердили: 56 монет.
